# Zebrafish: GitHub + scripts (API + SRA Toolkit)

We keep the **same repo** locally and on `sequoia:/home/zebrafish` so everyone runs the same scripts.
We use **two download approaches**: API for metadata/coordination, and SRA Toolkit for FASTQ generation.


## Why we keep both approaches

API = fast exploration + reproducible run lists; SRA Toolkit = standard implementation for producing FASTQs (fastq-dump / fasterq-dump).


## 0) Confirm local vs server repo are in sync

This prints the git commit hash locally and on the server (`/home/zebrafish`). If SSH fails, run `ssh pzg8794@sequoia.rit.edu` in a terminal once to set up keys/hostkey.



In [37]:
!ssh -o StrictHostKeyChecking=accept-new -o BatchMode=yes -o ConnectTimeout=10 -o ServerAliveInterval=5 -o ServerAliveCountMax=1 -n pzg8794@sequoia.rit.edu "hostname; whoami; pwd"


RIT information technology resources are for the use of the RIT community only. 
By using RIT information technology resources you acknowledge that you have read 
and comply with RIT's Code of Conduct for Computer and Network Use and RIT's 
Information Security Policy and Standards. Use of RIT information technology 
resources may be monitored and unauthorized use is strictly prohibited.
sequoia
pzg8794
/home/pzg8794


Checks SSH connectivity to the server (should print hostname, username, and a working directory).


In [38]:
%%bash
set -euo pipefail

# Compare local vs server git HEAD
GIT_ROOT="$(git rev-parse --show-toplevel)"
cd "$GIT_ROOT"

echo "LOCAL HEAD:  $(git rev-parse HEAD)"
git status -sb || true

echo
SSH="ssh -o StrictHostKeyChecking=accept-new -o BatchMode=yes -o ConnectTimeout=10 -o ServerAliveInterval=5 -o ServerAliveCountMax=1 pzg8794@sequoia.rit.edu"
REMOTE_REPO="/home/zebrafish"
$SSH "REMOTE_REPO=$REMOTE_REPO bash -s" <<'EOF'
set -euo pipefail
cd "$REMOTE_REPO"

echo "SERVER HEAD: $(git rev-parse HEAD)"
git status -sb || true
EOF


LOCAL HEAD:  63064b890ba6ffa69483294dd26e7126042bfc6f


## main...origin/main
 M zebrafish/zebrafish_github_setup_and_script_walkthrough.ipynb



RIT information technology resources are for the use of the RIT community only. 
By using RIT information technology resources you acknowledge that you have read 
and comply with RIT's Code of Conduct for Computer and Network Use and RIT's 
Information Security Policy and Standards. Use of RIT information technology 
resources may be monitored and unauthorized use is strictly prohibited.


SERVER HEAD: 63064b890ba6ffa69483294dd26e7126042bfc6f
## main...origin/main


Compares local vs server git commit (`HEAD`) so you know both environments are using the same code.


## 1) Pull the latest repo on the server (fixes missing scripts)

Run this once when things look out of date or you see a “missing file” error.


In [39]:
%%bash
set -euo pipefail

# Update server repo
SSH="ssh -o StrictHostKeyChecking=accept-new -o BatchMode=yes -o ConnectTimeout=10 -o ServerAliveInterval=5 -o ServerAliveCountMax=1 pzg8794@sequoia.rit.edu"
REMOTE_REPO="/home/zebrafish"

$SSH "REMOTE_REPO=$REMOTE_REPO bash -s" <<'EOF'
set -euo pipefail
cd "$REMOTE_REPO"

# If git refuses due to 'dubious ownership', run once:
#   git config --global --add safe.directory /home/zebrafish

git fetch origin
# Use ff-only to avoid accidental merges on the shared server

git pull --ff-only

echo "SERVER HEAD: $(git rev-parse HEAD)"
EOF


RIT information technology resources are for the use of the RIT community only. 
By using RIT information technology resources you acknowledge that you have read 
and comply with RIT's Code of Conduct for Computer and Network Use and RIT's 
Information Security Policy and Standards. Use of RIT information technology 
resources may be monitored and unauthorized use is strictly prohibited.


Already up to date.
SERVER HEAD: 63064b890ba6ffa69483294dd26e7126042bfc6f


Updates the shared server clone in `/home/zebrafish` with a fast-forward-only `git pull`.


## 2) Choose runs for team download

Pick runs by (a) a number of runs, (b) a runs file, or (c) an inline list; we split this once for the team.


In [45]:
%%bash
set -euo pipefail

# Build a single team run list (server) that everyone will use.
# Then split that list among piter/nikhi/samuel (no per-member filtering).

SSH="ssh -o StrictHostKeyChecking=accept-new -o BatchMode=yes -o ConnectTimeout=10 -o ServerAliveInterval=5 -o ServerAliveCountMax=1 pzg8794@sequoia.rit.edu"
REMOTE_REPO="/home/zebrafish"

ACC="PRJNA1277581"

# Choose ONE input method (server-side):
N_RUNS=0                  # 0 = use all runs from runs.all.txt
RUNS_FILE=""              # e.g., zebrafish/metadata/$ACC/runs.some_list.txt
RUNS_INLINE=""            # e.g., "SRR123 SRR456 SRR789" (space-separated)

$SSH "REMOTE_REPO=$REMOTE_REPO ACC=$ACC N_RUNS=$N_RUNS RUNS_FILE=$RUNS_FILE RUNS_INLINE=$RUNS_INLINE bash -s" <<'EOF'
set -euo pipefail
cd "$REMOTE_REPO"

ACC="$ACC"
N_RUNS="$N_RUNS"
RUNS_FILE="$RUNS_FILE"
RUNS_INLINE="$RUNS_INLINE"

ALL="zebrafish/metadata/$ACC/runs.all.txt"
TEAM="zebrafish/metadata/$ACC/runs.team.txt"
SPLITS_DIR="zebrafish/metadata/$ACC/splits"
mkdir -p "$(dirname "$TEAM")" "$SPLITS_DIR"

if [ -n "$RUNS_INLINE" ]; then
  echo "$RUNS_INLINE" | tr ' ' $'
' | sed '/^$/d' > "$TEAM"
  echo "TEAM runs from inline list -> $TEAM"
elif [ -n "$RUNS_FILE" ]; then
  [ -f "$RUNS_FILE" ] || { echo "ERROR: RUNS_FILE not found: $RUNS_FILE"; exit 2; }
  cp "$RUNS_FILE" "$TEAM"
  echo "TEAM runs from file ($RUNS_FILE) -> $TEAM"
else
  [ -f "$ALL" ] || { echo "ERROR: missing $ALL (run the RunInfo step first)"; exit 2; }
  if [ "$N_RUNS" != "0" ]; then
    head -n "$N_RUNS" "$ALL" > "$TEAM"
    echo "TEAM runs = first $N_RUNS from $ALL -> $TEAM"
  else
    cp "$ALL" "$TEAM"
    echo "TEAM runs = all runs from $ALL -> $TEAM"
  fi
fi

echo
echo "TEAM run count + preview:"
wc -l "$TEAM" || true
head -n 5 "$TEAM" || true

python3 zebrafish/scripts/split_runs_among_members.py   --runs-file "$TEAM"   --members piter nikhi samuel   --out-dir "$SPLITS_DIR"   --prefix runs.member

echo
echo "Split files:"
ls -la "$SPLITS_DIR"
EOF


RIT information technology resources are for the use of the RIT community only. 
By using RIT information technology resources you acknowledge that you have read 
and comply with RIT's Code of Conduct for Computer and Network Use and RIT's 
Information Security Policy and Standards. Use of RIT information technology 
resources may be monitored and unauthorized use is strictly prohibited.


TEAM runs = all runs from zebrafish/metadata/PRJNA1277581/runs.all.txt -> zebrafish/metadata/PRJNA1277581/runs.team.txt

TEAM run count + preview:
30 zebrafish/metadata/PRJNA1277581/runs.team.txt
SRR34002439
SRR34002438
SRR34002437
SRR34002436
SRR34002435
piter: 10 -> zebrafish/metadata/PRJNA1277581/splits/runs.member.piter.txt
nikhi: 10 -> zebrafish/metadata/PRJNA1277581/splits/runs.member.nikhi.txt
samuel: 10 -> zebrafish/metadata/PRJNA1277581/splits/runs.member.samuel.txt

Split files:
total 24
drwxrwxr-x 2 pzg8794 pzg8794 4096 Feb 18 21:51 .
drwxrwxr-x 3 pzg8794 pzg8794 4096 Feb 18 21:51 ..
-rw-rw-r-- 1 pzg8794 pzg8794  120 Feb 18 23:23 runs.member.nikhi.txt
-rw-rw-r-- 1 pzg8794 pzg8794   12 Feb 18 21:51 runs.member.piter.first1.txt
-rw-rw-r-- 1 pzg8794 pzg8794  120 Feb 18 23:23 runs.member.piter.txt
-rw-rw-r-- 1 pzg8794 pzg8794  120 Feb 18 23:23 runs.member.samuel.txt


Creates `runs.team.txt` (team-wide) then splits it into per-member run lists under `.../splits/`.


## 3) What scripts exist (shared by all members)

These are the scripts everyone should use (local or on the server).


In [40]:
%%bash
set -euo pipefail

# List scripts locally (this notebook) + on server.

echo "LOCAL scripts (cwd=$(pwd))"
ls -la scripts | sed -n '1,200p'

echo
echo "SERVER scripts (/home/zebrafish/zebrafish/scripts)"
SSH="ssh -o StrictHostKeyChecking=accept-new -o BatchMode=yes -o ConnectTimeout=10 -o ServerAliveInterval=5 -o ServerAliveCountMax=1 pzg8794@sequoia.rit.edu"
REMOTE_REPO="/home/zebrafish"

if ! $SSH "REMOTE_REPO=$REMOTE_REPO bash -s" <<'EOF'
set -euo pipefail
cd "$REMOTE_REPO"
ls -la zebrafish/scripts | sed -n '1,200p'
EOF
then
  echo
  echo "SSH failed (or server path missing). Run this once in a terminal to fix auth/hostkey:"
  echo "  ssh pzg8794@sequoia.rit.edu"
fi


LOCAL scripts (cwd=/Users/pitergarcia/DataScience/Semester5/BIOL550/group_project/zebrafish)
total 96
drwxr-xr-x@ 11 pitergarcia  staff   352 Feb 18 23:04 .
drwxr-xr-x@ 12 pitergarcia  staff   384 Feb 18 23:21 ..
drwxr-xr-x@  7 pitergarcia  staff   224 Feb 18 22:33 __pycache__
-rw-r--r--@  1 pitergarcia  staff  2240 Feb 18 17:01 compare_fastq_subsets.py
-rwxr-xr-x@  1 pitergarcia  staff  4285 Feb 18 23:21 download_fastq_sratoolkit.sh
-rwxr-xr-x@  1 pitergarcia  staff  2395 Feb 18 21:41 download_fastq_sratoolkit_from_runs.sh
-rw-r--r--@  1 pitergarcia  staff  3844 Feb 18 17:30 download_runfiles_ncbi_download_path.py
-rwxr-xr-x@  1 pitergarcia  staff  3491 Feb 18 21:41 download_test_5_runs_fastq.sh
-rw-r--r--@  1 pitergarcia  staff  4800 Feb 18 16:58 ensure_sratoolkit.py
-rw-r--r--@  1 pitergarcia  staff  9869 Feb 18 11:28 get_zebrafish_data_sra.py
-rw-r--r--@  1 pitergarcia  staff  1833 Feb 18 17:31 split_runs_among_members.py

SERVER scripts (/home/zebrafish/zebrafish/scripts)


RIT information technology resources are for the use of the RIT community only. 
By using RIT information technology resources you acknowledge that you have read 
and comply with RIT's Code of Conduct for Computer and Network Use and RIT's 
Information Security Policy and Standards. Use of RIT information technology 
resources may be monitored and unauthorized use is strictly prohibited.


total 56
drwxrwxr-x 2 pzg8794 pzg8794 4096 Feb 18 23:22 .
drwxrwxr-x 7 pzg8794 pzg8794 4096 Feb 18 23:15 ..
-rw-rw-r-- 1 pzg8794 pzg8794 2240 Feb 18 17:39 compare_fastq_subsets.py
-rwxrwxr-x 1 pzg8794 pzg8794 2395 Feb 18 23:22 download_fastq_sratoolkit_from_runs.sh
-rwxrwxr-x 1 pzg8794 pzg8794 4285 Feb 18 23:22 download_fastq_sratoolkit.sh
-rw-rw-r-- 1 pzg8794 pzg8794 3844 Feb 18 17:39 download_runfiles_ncbi_download_path.py
-rwxrwxr-x 1 pzg8794 pzg8794 3491 Feb 18 21:50 download_test_5_runs_fastq.sh
-rw-rw-r-- 1 pzg8794 pzg8794 4800 Feb 18 17:39 ensure_sratoolkit.py
-rw-rw-r-- 1 pzg8794 pzg8794 9869 Feb 18 15:24 get_zebrafish_data_sra.py
-rw-rw-r-- 1 pzg8794 pzg8794 1833 Feb 18 17:39 split_runs_among_members.py


Lists the scripts locally and on the server so the team knows what is available and in-sync.


## 4) API approach: metadata + coordination

We use the SRA RunInfo API to get run metadata and generate stable SRR lists for the team.


### A1) Script: `get_zebrafish_data_sra.py`

Fetches RunInfo (`runinfo.csv`) and writes SRR lists (all + filtered) under `zebrafish/metadata/<ACC>/`.


In [41]:
%%bash
set -euo pipefail

# Show script help (server)
SSH="ssh -o StrictHostKeyChecking=accept-new -o BatchMode=yes -o ConnectTimeout=10 -o ServerAliveInterval=5 -o ServerAliveCountMax=1 pzg8794@sequoia.rit.edu"
REMOTE_REPO="/home/zebrafish"

$SSH "REMOTE_REPO=$REMOTE_REPO bash -s" <<'EOF'
set -euo pipefail
cd "$REMOTE_REPO"
python3 zebrafish/scripts/get_zebrafish_data_sra.py --help | sed -n '1,120p'
EOF


RIT information technology resources are for the use of the RIT community only. 
By using RIT information technology resources you acknowledge that you have read 
and comply with RIT's Code of Conduct for Computer and Network Use and RIT's 
Information Security Policy and Standards. Use of RIT information technology 
resources may be monitored and unauthorized use is strictly prohibited.


usage: get_zebrafish_data_sra.py [-h] [--acc ACC] [--out-dir OUT_DIR]
                                 [--organism ORGANISM]
                                 [--library-strategy LIBRARY_STRATEGY]
                                 [--library-layout LIBRARY_LAYOUT]
                                 [--min-spots MIN_SPOTS]
                                 [--min-avg-length MIN_AVG_LENGTH]
                                 [--max-runs MAX_RUNS] [--write-download-urls]

Fetch SRA RunInfo and SRR lists for zebrafish (Danio rerio) RNA-seq data.

options:
  -h, --help            show this help message and exit
  --acc ACC             SRA query/accession (e.g., PRJNA..., SRP..., SRX...).
  --out-dir OUT_DIR     Output directory.
  --organism ORGANISM   ScientificName filter (exact match).
  --library-strategy LIBRARY_STRATEGY
                        LibraryStrategy filter (exact match).
  --library-layout LIBRARY_LAYOUT
                        LibraryLayout filter (exact match).
  --min-spots MIN_

Shows help for the API metadata script that generates `runinfo.csv` and run lists used by the team.


### A2) Script: `download_runfiles_ncbi_download_path.py`

Downloads the run file for each SRR using the `download_path` column in `runinfo.csv` (no SRA Toolkit needed).


In [42]:
%%bash
set -euo pipefail

# Show script help (server)
SSH="ssh -o StrictHostKeyChecking=accept-new -o BatchMode=yes -o ConnectTimeout=10 -o ServerAliveInterval=5 -o ServerAliveCountMax=1 pzg8794@sequoia.rit.edu"
REMOTE_REPO="/home/zebrafish"

$SSH "REMOTE_REPO=$REMOTE_REPO bash -s" <<'EOF'
set -euo pipefail
cd "$REMOTE_REPO"
python3 zebrafish/scripts/download_runfiles_ncbi_download_path.py --help | sed -n '1,160p'
EOF


RIT information technology resources are for the use of the RIT community only. 
By using RIT information technology resources you acknowledge that you have read 
and comply with RIT's Code of Conduct for Computer and Network Use and RIT's 
Information Security Policy and Standards. Use of RIT information technology 
resources may be monitored and unauthorized use is strictly prohibited.


usage: download_runfiles_ncbi_download_path.py [-h] --acc ACC
                                               --runs-file RUNS_FILE
                                               --runinfo-csv RUNINFO_CSV
                                               [--base-dir BASE_DIR] [--force]

Download SRA run files using NCBI RunInfo `download_path` URLs (no SRA Toolkit
required). Input: a run list file + runinfo.csv.

options:
  -h, --help            show this help message and exit
  --acc ACC             BioProject / query label (used only for path
                        defaults).
  --runs-file RUNS_FILE
                        Text file with one SRR per line.
  --runinfo-csv RUNINFO_CSV
                        SRA runinfo.csv containing a `download_path` column.
  --base-dir BASE_DIR   Base output directory (default:
                        zebrafish/data/runfiles).
  --force               Re-download even if files exist.


Shows help for the script that downloads SRA “runfiles” via NCBI RunInfo `download_path`.

Example (server):

```bash
python3 zebrafish/scripts/download_runfiles_ncbi_download_path.py \
  --acc PRJNA1277581 \
  --runinfo-csv zebrafish/metadata/PRJNA1277581/runinfo.csv \
  --runs-file  zebrafish/metadata/PRJNA1277581/runs.team.txt \
  --base-dir   zebrafish/data/runfiles
```


## 5) SRA Toolkit approach: FASTQs (implementation)

We install SRA Toolkit once, then use it to convert SRRs into paired FASTQ files for analysis.


### S1) Install toolkit: `ensure_sratoolkit.py` (or Step 4a in the other notebook)

This keeps the toolkit in `zebrafish/tools/sratoolkit/` (gitignored) so server + local usage matches.


In [43]:
%%bash
set -euo pipefail

# Install / ensure SRA Toolkit on the server
SSH="ssh -o StrictHostKeyChecking=accept-new -o BatchMode=yes -o ConnectTimeout=10 -o ServerAliveInterval=5 -o ServerAliveCountMax=1 pzg8794@sequoia.rit.edu"
REMOTE_REPO="/home/zebrafish"

$SSH "REMOTE_REPO=$REMOTE_REPO bash -s" <<'EOF'
set -euo pipefail
cd "$REMOTE_REPO"
python3 zebrafish/scripts/ensure_sratoolkit.py --repo-root "$PWD" --print-bin
EOF


RIT information technology resources are for the use of the RIT community only. 
By using RIT information technology resources you acknowledge that you have read 
and comply with RIT's Code of Conduct for Computer and Network Use and RIT's 
Information Security Policy and Standards. Use of RIT information technology 
resources may be monitored and unauthorized use is strictly prohibited.


/home/zebrafish/zebrafish/tools/sratoolkit/bin


Ensures SRA Toolkit exists under `zebrafish/tools/sratoolkit/` on the server and prints its `bin/` path.


### S2) Script: `download_fastq_sratoolkit.sh`

This is the **one wrapper** we use. It accepts runs as an inline list, a file, or `--member + --n-runs` (first N from your assigned file).

Under the hood it calls `download_fastq_sratoolkit_from_runs.sh` (the core loop that runs `prefetch` + `fasterq-dump`).


In [44]:
%%bash
set -euo pipefail

# Show script help (server)
SSH="ssh -o StrictHostKeyChecking=accept-new -o BatchMode=yes -o ConnectTimeout=10 -o ServerAliveInterval=5 -o ServerAliveCountMax=1 pzg8794@sequoia.rit.edu"
REMOTE_REPO="/home/zebrafish"

$SSH "REMOTE_REPO=$REMOTE_REPO bash -s" <<'EOF'
set -euo pipefail
cd "$REMOTE_REPO"

bash zebrafish/scripts/download_fastq_sratoolkit.sh --help | sed -n '1,200p'
EOF


RIT information technology resources are for the use of the RIT community only. 
By using RIT information technology resources you acknowledge that you have read 
and comply with RIT's Code of Conduct for Computer and Network Use and RIT's 
Information Security Policy and Standards. Use of RIT information technology 
resources may be monitored and unauthorized use is strictly prohibited.


Download FASTQs for SRR runs using SRA Toolkit (prefetch + fasterq-dump).

This is the ONE wrapper you should use. Provide runs in exactly one of 3 ways:
  A) Inline list:  --runs "SRR... SRR..."
  B) File:         --runs-file path/to/runs.txt
  C) Member + N:   --member <name> --n-runs N
                  (takes the first N SRRs from your assigned file:
                   metadata/<ACC>/splits/runs.member.<member>.txt)

Recommended (start with 1 run):
  bash scripts/download_fastq_sratoolkit.sh \
    --acc PRJNA1277581 \
    --member piter \
    --n-runs 1 \
    --out-dir data/PRJNA1277581 \
    --threads 4

Other examples:
  # Use a file directly
  bash scripts/download_fastq_sratoolkit.sh \
    --acc PRJNA1277581 \
    --runs-file metadata/PRJNA1277581/splits/runs.member.piter.txt \
    --out-dir data/PRJNA1277581 \
    --threads 4

  # Use an inline list
  bash scripts/download_fastq_sratoolkit.sh \
    --acc PRJNA1277581 \
    --runs "SRR34002427 SRR34002428" \
    --out-dir data/

Shows help for the SRA Toolkit FASTQ downloader script (takes a runs file + output directory).

Examples (server):

```bash
# Team-wide download
bash zebrafish/scripts/download_fastq_sratoolkit.sh \
  --runs-file zebrafish/metadata/PRJNA1277581/runs.team.txt \
  --out-dir   zebrafish/data/PRJNA1277581/team \
  --threads   4

# Per-member download (after Step 3 split)
bash zebrafish/scripts/download_fastq_sratoolkit.sh \
  --runs-file zebrafish/metadata/PRJNA1277581/splits/runs.member.piter.txt \
  --out-dir   zebrafish/data/PRJNA1277581 \
  --threads   4
```


### Wrapper: one command that runs SRA Toolkit

Our FASTQ download is just the professor’s workflow automated for a **list of SRR accessions**. The wrapper script is `zebrafish/scripts/download_fastq_sratoolkit.sh` and it runs the same two SRA Toolkit commands for each SRR:

### 1) `prefetch <SRR>` (download the run into the SRA cache)
- What it does: downloads the run file for an accession (an `.sra`-style runfile) into the SRA Toolkit cache (typically under `~/.ncbi/public/sra/`, depending on toolkit config).
- Why we use it: it’s **resumable** and keeps the “download” step separate from conversion, so if conversion fails you don’t re-download everything.

### 2) `fasterq-dump ... <SRR>` (convert runfile → FASTQ)
In our wrapper we call:

```bash
fasterq-dump --split-files --threads <N> --outdir <RUN_DIR> <SRR>
```

- `--split-files`: for paired-end runs, writes two files: `<SRR>_1.fastq` and `<SRR>_2.fastq`.
- `--threads <N>`: uses multiple threads to speed up conversion.
- `--outdir <RUN_DIR>`: writes outputs into a per-run folder so files from different SRRs never collide.

### 3) `gzip -f` (compress FASTQs)
`fasterq-dump` writes **uncompressed** `.fastq` by default, so we immediately compress:

```bash
gzip -f <SRR>_1.fastq <SRR>_2.fastq
```

That produces the final files we keep:
- `<SRR>_1.fastq.gz`
- `<SRR>_2.fastq.gz`

### How the wrapper script ties it together
For each SRR in `--runs-file`, the script:
1. Creates `--out-dir/<SRR>/`
2. **Skips** the SRR if `<SRR>_1.fastq.gz` and `<SRR>_2.fastq.gz` already exist (unless `--force` is used)
3. Runs `prefetch <SRR>`
4. Runs `fasterq-dump --split-files --threads ... --outdir ... <SRR>`
5. Runs `gzip -f` on the two FASTQs

This gives us a reproducible, team-friendly pattern: everyone runs the same script with a different runs list (or split list), but the underlying commands are exactly the professor’s.

### Quick note: `fastq-dump` vs `fasterq-dump`
- For **full downloads**, we prefer `prefetch + fasterq-dump` because it’s the modern/fast path.
- In the *other* notebook where we extract a **spot-range subset**, we use `fastq-dump -N/-X` because `fastq-dump` supports spot subsetting while the toolkit build we have may not support `-N/-X` on `fasterq-dump`.


## Member download: piter

This section runs the **ONE wrapper** `zebrafish/scripts/download_fastq_sratoolkit.sh` on `sequoia` (via SSH). It downloads FASTQs for the first `N_RUNS` SRRs from your assigned list `zebrafish/metadata/<ACC>/splits/runs.member.piter.txt` and writes them to `zebrafish/data/<ACC>/<SRR>/` as `<SRR>_1.fastq.gz` and `<SRR>_2.fastq.gz`.


In [49]:
%%bash
set -euo pipefail

# piter: download FASTQs on the server (SRA Toolkit)
SSH="ssh -o StrictHostKeyChecking=accept-new -o BatchMode=yes -o ConnectTimeout=10 -o ServerAliveInterval=5 -o ServerAliveCountMax=1 pzg8794@sequoia.rit.edu"
REMOTE_REPO="/home/zebrafish"

ACC="PRJNA1277581"
THREADS=4

# Recommended: start with 1 run, then change to 2, 3, ... when ready
N_RUNS=5

$SSH "REMOTE_REPO=$REMOTE_REPO ACC=$ACC THREADS=$THREADS N_RUNS=$N_RUNS MEMBER=piter bash -s" <<'EOF'
set -euo pipefail
cd "$REMOTE_REPO"

# Make sure the repo-bundled SRA Toolkit is on PATH
export PATH="$PWD/zebrafish/tools/sratoolkit/bin:$PATH"

OUT_DIR="zebrafish/data/$ACC"

# (1) Download the first N_RUNS SRRs from your assigned list (simple + recommended)
bash zebrafish/scripts/download_fastq_sratoolkit.sh   --acc "$ACC"   --member "$MEMBER"   --n-runs "$N_RUNS"   --out-dir "$OUT_DIR"   --threads "$THREADS"

# --- Other options (keep commented until you need them) ---

# (2) Download from your FULL assigned list:
# RUNS_FILE="zebrafish/metadata/$ACC/splits/runs.member.${MEMBER}.txt"
# bash zebrafish/scripts/download_fastq_sratoolkit.sh #   --acc "$ACC" #   --runs-file "$RUNS_FILE" #   --out-dir "$OUT_DIR" #   --threads "$THREADS"

# (3) Download from an inline list (space-separated SRRs):
# bash zebrafish/scripts/download_fastq_sratoolkit.sh #   --acc "$ACC" #   --runs "SRR34002427 SRR34002428" #   --out-dir "$OUT_DIR" #   --threads "$THREADS"

# (4) Download from any runs file you provide:
# RUNS_FILE_ANY="zebrafish/metadata/$ACC/runs.some_list.txt"
# bash zebrafish/scripts/download_fastq_sratoolkit.sh #   --acc "$ACC" #   --runs-file "$RUNS_FILE_ANY" #   --out-dir "$OUT_DIR" #   --threads "$THREADS"
EOF


Python(71583) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
RIT information technology resources are for the use of the RIT community only. 
By using RIT information technology resources you acknowledge that you have read 
and comply with RIT's Code of Conduct for Computer and Network Use and RIT's 
Information Security Policy and Standards. Use of RIT information technology 
resources may be monitored and unauthorized use is strictly prohibited.


acc:      PRJNA1277581
out_dir:  zebrafish/data/PRJNA1277581
threads:  4
force:    0
member:   piter
n_runs:   5

These SRRs will be downloaded:
SRR34002439
SRR34002436
SRR34002433
SRR34002430
SRR34002427

$ bash $ /home/zebrafish/zebrafish/scripts/download_fastq_sratoolkit_from_runs.sh $ --runs-file $ /tmp/runs.PRJNA1277581.piter.first5.vYPYdS.txt $ --out-dir $ zebrafish/data/PRJNA1277581 $ --threads $ 4 
runs_file: /tmp/runs.PRJNA1277581.piter.first5.vYPYdS.txt
out_dir:   zebrafish/data/PRJNA1277581
threads:   4
force:     0

skip (exists): SRR34002439
skip (exists): SRR34002436

== SRR34002433 ==
2026-02-19T09:02:46 prefetch.3.3.0: 1) Resolving 'SRR34002433'...
2026-02-19T09:02:47 prefetch.3.3.0: Current preference is set to retrieve SRA Normalized Format files with full base quality scores
2026-02-19T09:02:47 prefetch.3.3.0: 1) Downloading 'SRR34002433'...
2026-02-19T09:02:47 prefetch.3.3.0:  SRA Normalized Format file is being retrieved
2026-02-19T09:02:47 prefetch.3.3.0:  Downloa

spots read      : 113,034,961
reads read      : 226,069,922
reads written   : 226,069,922
Timeout, server sequoia.rit.edu not responding.


CalledProcessError: Command 'b'set -euo pipefail\n\n# piter: download FASTQs on the server (SRA Toolkit)\nSSH="ssh -o StrictHostKeyChecking=accept-new -o BatchMode=yes -o ConnectTimeout=10 -o ServerAliveInterval=5 -o ServerAliveCountMax=1 pzg8794@sequoia.rit.edu"\nREMOTE_REPO="/home/zebrafish"\n\nACC="PRJNA1277581"\nTHREADS=4\n\n# Recommended: start with 1 run, then change to 2, 3, ... when ready\nN_RUNS=5\n\n$SSH "REMOTE_REPO=$REMOTE_REPO ACC=$ACC THREADS=$THREADS N_RUNS=$N_RUNS MEMBER=piter bash -s" <<\'EOF\'\nset -euo pipefail\ncd "$REMOTE_REPO"\n\n# Make sure the repo-bundled SRA Toolkit is on PATH\nexport PATH="$PWD/zebrafish/tools/sratoolkit/bin:$PATH"\n\nOUT_DIR="zebrafish/data/$ACC"\n\n# (1) Download the first N_RUNS SRRs from your assigned list (simple + recommended)\nbash zebrafish/scripts/download_fastq_sratoolkit.sh   --acc "$ACC"   --member "$MEMBER"   --n-runs "$N_RUNS"   --out-dir "$OUT_DIR"   --threads "$THREADS"\n\n# --- Other options (keep commented until you need the

Runs a **small smoke test** for Piter: set `N_RUNS=1` first, confirm it works, then increase to `2`, `3`, ... . Re-running is safe: it will print `skip (exists)` for runs already downloaded.


## Member download: nikhi

This section runs the **ONE wrapper** `zebrafish/scripts/download_fastq_sratoolkit.sh` on `sequoia` (via SSH). It downloads FASTQs for the first `N_RUNS` SRRs from your assigned list `zebrafish/metadata/<ACC>/splits/runs.member.nikhi.txt` and writes them to `zebrafish/data/<ACC>/<SRR>/` as `<SRR>_1.fastq.gz` and `<SRR>_2.fastq.gz`.


In [ ]:
%%bash
set -euo pipefail

# nikhi: download FASTQs on the server (SRA Toolkit)
SSH="ssh -o StrictHostKeyChecking=accept-new -o BatchMode=yes -o ConnectTimeout=10 -o ServerAliveInterval=5 -o ServerAliveCountMax=1 pzg8794@sequoia.rit.edu"
REMOTE_REPO="/home/zebrafish"

ACC="PRJNA1277581"
THREADS=4

# Recommended: start with 1 run, then change to 2, 3, ... when ready
N_RUNS=1

$SSH "REMOTE_REPO=$REMOTE_REPO ACC=$ACC THREADS=$THREADS N_RUNS=$N_RUNS MEMBER=nikhi bash -s" <<'EOF'
set -euo pipefail
cd "$REMOTE_REPO"

# Make sure the repo-bundled SRA Toolkit is on PATH
export PATH="$PWD/zebrafish/tools/sratoolkit/bin:$PATH"

OUT_DIR="zebrafish/data/$ACC"

# (1) Download the first N_RUNS SRRs from your assigned list (simple + recommended)
bash zebrafish/scripts/download_fastq_sratoolkit.sh   --acc "$ACC"   --member "$MEMBER"   --n-runs "$N_RUNS"   --out-dir "$OUT_DIR"   --threads "$THREADS"

# --- Other options (keep commented until you need them) ---

# (2) Download from your FULL assigned list:
# RUNS_FILE="zebrafish/metadata/$ACC/splits/runs.member.${MEMBER}.txt"
# bash zebrafish/scripts/download_fastq_sratoolkit.sh #   --acc "$ACC" #   --runs-file "$RUNS_FILE" #   --out-dir "$OUT_DIR" #   --threads "$THREADS"

# (3) Download from an inline list (space-separated SRRs):
# bash zebrafish/scripts/download_fastq_sratoolkit.sh #   --acc "$ACC" #   --runs "SRR34002427 SRR34002428" #   --out-dir "$OUT_DIR" #   --threads "$THREADS"

# (4) Download from any runs file you provide:
# RUNS_FILE_ANY="zebrafish/metadata/$ACC/runs.some_list.txt"
# bash zebrafish/scripts/download_fastq_sratoolkit.sh #   --acc "$ACC" #   --runs-file "$RUNS_FILE_ANY" #   --out-dir "$OUT_DIR" #   --threads "$THREADS"
EOF


Runs a **small smoke test** for Nikhi: set `N_RUNS=1` first, confirm it works, then increase to `2`, `3`, ... . Re-running is safe: it will print `skip (exists)` for runs already downloaded.


## Member download: samuel

This section runs the **ONE wrapper** `zebrafish/scripts/download_fastq_sratoolkit.sh` on `sequoia` (via SSH). It downloads FASTQs for the first `N_RUNS` SRRs from your assigned list `zebrafish/metadata/<ACC>/splits/runs.member.samuel.txt` and writes them to `zebrafish/data/<ACC>/<SRR>/` as `<SRR>_1.fastq.gz` and `<SRR>_2.fastq.gz`.


In [ ]:
%%bash
set -euo pipefail

# samuel: download FASTQs on the server (SRA Toolkit)
SSH="ssh -o StrictHostKeyChecking=accept-new -o BatchMode=yes -o ConnectTimeout=10 -o ServerAliveInterval=5 -o ServerAliveCountMax=1 pzg8794@sequoia.rit.edu"
REMOTE_REPO="/home/zebrafish"

ACC="PRJNA1277581"
THREADS=4

# Recommended: start with 1 run, then change to 2, 3, ... when ready
N_RUNS=1

$SSH "REMOTE_REPO=$REMOTE_REPO ACC=$ACC THREADS=$THREADS N_RUNS=$N_RUNS MEMBER=samuel bash -s" <<'EOF'
set -euo pipefail
cd "$REMOTE_REPO"

# Make sure the repo-bundled SRA Toolkit is on PATH
export PATH="$PWD/zebrafish/tools/sratoolkit/bin:$PATH"

OUT_DIR="zebrafish/data/$ACC"

# (1) Download the first N_RUNS SRRs from your assigned list (simple + recommended)
bash zebrafish/scripts/download_fastq_sratoolkit.sh   --acc "$ACC"   --member "$MEMBER"   --n-runs "$N_RUNS"   --out-dir "$OUT_DIR"   --threads "$THREADS"

# --- Other options (keep commented until you need them) ---

# (2) Download from your FULL assigned list:
# RUNS_FILE="zebrafish/metadata/$ACC/splits/runs.member.${MEMBER}.txt"
# bash zebrafish/scripts/download_fastq_sratoolkit.sh #   --acc "$ACC" #   --runs-file "$RUNS_FILE" #   --out-dir "$OUT_DIR" #   --threads "$THREADS"

# (3) Download from an inline list (space-separated SRRs):
# bash zebrafish/scripts/download_fastq_sratoolkit.sh #   --acc "$ACC" #   --runs "SRR34002427 SRR34002428" #   --out-dir "$OUT_DIR" #   --threads "$THREADS"

# (4) Download from any runs file you provide:
# RUNS_FILE_ANY="zebrafish/metadata/$ACC/runs.some_list.txt"
# bash zebrafish/scripts/download_fastq_sratoolkit.sh #   --acc "$ACC" #   --runs-file "$RUNS_FILE_ANY" #   --out-dir "$OUT_DIR" #   --threads "$THREADS"
EOF


Runs a **small smoke test** for Samuel: set `N_RUNS=1` first, confirm it works, then increase to `2`, `3`, ... . Re-running is safe: it will print `skip (exists)` for runs already downloaded.
